# Zero — Phase 1: Baseline Evaluation
> Establishing the reasoning floor across Qwen2.5-1.5B, 3B, and 7B with zero training.

**What this notebook does:**
- Runs all three base models against 16 text-only CTF problems (4 per category)
- Scores each response: flag correctness + directness
- Outputs a CSV for analysis

**No training. No fine-tuning. Just the floor.**

In [ ]:
# Cell 1 — Install dependencies
!pip install transformers accelerate bitsandbytes sentence-transformers -q

In [ ]:
# Cell 2 — Imports
import torch
import re
import json
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, util
from datetime import datetime

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name} — {props.total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 3 — Benchmark problems
# 16 text-only CTF problems, 4 per category.
# All self-contained — no binary attachments, no external files.
# answer_type: 'exact' (string match) or 'semantic' (embedding similarity)

BENCHMARK = [

    # ── WEB EXPLOITATION ────────────────────────────────────────────────────
    {
        "id": "web_001",
        "category": "web",
        "difficulty": "easy",
        "prompt": """A login form POSTs username and password to /login. The backend builds its query like this:

query = \"SELECT * FROM users WHERE username='\" + username + \"' AND password='\" + password + \"'\"

There is no sanitization. Give the exact username field payload that logs you in as the first user in the database without knowing any credentials.""",
        "answer": "' OR '1'='1",
        "answer_type": "exact",
    },
    {
        "id": "web_002",
        "category": "web",
        "difficulty": "easy",
        "prompt": """An HTTP response contains this header:

Set-Cookie: role=dXNlcg==; Path=/

The /admin endpoint returns 403 unless the role cookie decodes to 'admin'. What value should you set the cookie to?""",
        "answer": "YWRtaW4=",
        "answer_type": "exact",
    },
    {
        "id": "web_003",
        "category": "web",
        "difficulty": "medium",
        "prompt": """A page renders user input directly into HTML:

<div class=\"comment\">USER_INPUT_HERE</div>

The application strips <script> tags but does nothing else. Give a payload that executes alert(1) without using a script tag.""",
        "answer": "<img src=x onerror=alert(1)>",
        "answer_type": "semantic",
    },
    {
        "id": "web_004",
        "category": "web",
        "difficulty": "medium",
        "prompt": """A file download endpoint works like this:

GET /download?file=report.pdf

The server reads: open('/var/www/files/' + request.args['file'])

There is no path sanitization. Give the exact payload to read /etc/passwd.""",
        "answer": "../../../etc/passwd",
        "answer_type": "exact",
    },

    # ── CRYPTOGRAPHY ────────────────────────────────────────────────────────
    {
        "id": "crypto_001",
        "category": "crypto",
        "difficulty": "easy",
        "prompt": """Decrypt this Caesar cipher. The shift is unknown but the plaintext is an English sentence.

Ciphertext: KHOOR ZRUOG

What is the plaintext?""",
        "answer": "HELLO WORLD",
        "answer_type": "semantic",
    },
    {
        "id": "crypto_002",
        "category": "crypto",
        "difficulty": "easy",
        "prompt": """Decode this string. It looks like Base64.

ZmxhZ3tiYXNlNjRfaXNfbm90X2VuY3J5cHRpb259

What does it decode to?""",
        "answer": "flag{base64_is_not_encryption}",
        "answer_type": "exact",
    },
    {
        "id": "crypto_003",
        "category": "crypto",
        "difficulty": "medium",
        "prompt": """You intercepted two ciphertexts encrypted with the same XOR key (key reuse):

C1 = 1a0e0a1c4b
C2 = 0e1b1c0f5e

You know C2 decrypts to 'hello'. What does C1 decrypt to? Show your reasoning.""",
        "answer": "4corg",
        "answer_type": "semantic",
    },
    {
        "id": "crypto_004",
        "category": "crypto",
        "difficulty": "medium",
        "prompt": """RSA with small parameters:

n = 3233
e = 17
ciphertext = 2790

Factor n, compute the private key d, and decrypt the ciphertext. What is the plaintext integer?""",
        "answer": "65",
        "answer_type": "exact",
    },

    # ── REVERSE ENGINEERING ─────────────────────────────────────────────────
    {
        "id": "re_001",
        "category": "re",
        "difficulty": "easy",
        "prompt": """This Python script is the checker for a CTF flag:

import hashlib
secret = [102, 108, 97, 103, 123, 114, 51, 118, 51, 114, 115, 51, 125]
user_input = input('Flag: ')
if list(user_input.encode()) == secret:
    print('Correct!')

What is the flag?""",
        "answer": "flag{r3v3rs3}",
        "answer_type": "exact",
    },
    {
        "id": "re_002",
        "category": "re",
        "difficulty": "easy",
        "prompt": """Pseudocode from a crackme:

function check(input):
    key = 'ZEROK3Y'
    if len(input) != len(key):
        return False
    for i in range(len(key)):
        if ord(input[i]) ^ 0x41 != ord(key[i]) ^ 0x41:
            return False
    return True

What input makes check() return True?""",
        "answer": "ZEROK3Y",
        "answer_type": "exact",
    },
    {
        "id": "re_003",
        "category": "re",
        "difficulty": "medium",
        "prompt": """A binary check function in C pseudocode:

int check(char *input) {
    char enc[] = {0x66, 0x6c, 0x61, 0x67, 0x7b, 0x6c, 0x30, 0x67, 0x69, 0x63, 0x7d};
    for (int i = 0; i < 11; i++) {
        if (input[i] != enc[i]) return 0;
    }
    return 1;
}

What string does the function accept?""",
        "answer": "flag{l0gic}",
        "answer_type": "exact",
    },
    {
        "id": "re_004",
        "category": "re",
        "difficulty": "medium",
        "prompt": """A function transforms input before comparing:

def transform(s):
    return ''.join(chr(ord(c) + 3) if c.isalpha() else c for c in s)

target = 'iodj{fubswr}'

if transform(input_str) == target:
    print('Correct')

What is input_str?""",
        "answer": "flag{crypto}",
        "answer_type": "exact",
    },

    # ── FORENSICS / OSINT ───────────────────────────────────────────────────
    {
        "id": "forensics_001",
        "category": "forensics",
        "difficulty": "easy",
        "prompt": """A file starts with these bytes (hex):

89 50 4E 47 0D 0A 1A 0A

What file format is this, and what is the significance of the magic bytes?""",
        "answer": "PNG",
        "answer_type": "semantic",
    },
    {
        "id": "forensics_002",
        "category": "forensics",
        "difficulty": "easy",
        "prompt": """An image file's EXIF metadata contains:

GPS Latitude:  37.4220° N
GPS Longitude: 122.0841° W
Device: iPhone 13
Comment: the_flag_is_here

What is the flag hidden in the metadata?""",
        "answer": "the_flag_is_here",
        "answer_type": "exact",
    },
    {
        "id": "forensics_003",
        "category": "forensics",
        "difficulty": "medium",
        "prompt": """A packet capture shows this sequence of TCP payloads (ASCII):

Packet 1: GET /
Packet 2: Host: ctf.example.com
Packet 3: X-Flag: ZmxhZ3tuZXR3b3JrX2ZvcmVuc2ljc30=
Packet 4: Connection: close

Extract and decode the flag.""",
        "answer": "flag{network_forensics}",
        "answer_type": "exact",
    },
    {
        "id": "forensics_004",
        "category": "forensics",
        "difficulty": "medium",
        "prompt": """A deleted file was recovered from a disk image. The hex dump of the first 32 bytes is:

50 4B 03 04 14 00 00 00 08 00 00 00 00 00 00 00
00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00

What file format is this? The investigator needs to know which tool to use to extract its contents.""",
        "answer": "ZIP",
        "answer_type": "semantic",
    },
]

print(f"Benchmark loaded: {len(BENCHMARK)} problems")
for cat in ['web', 'crypto', 're', 'forensics']:
    n = sum(1 for p in BENCHMARK if p['category'] == cat)
    print(f"  {cat}: {n}")

In [ ]:
# Cell 4 — Scoring utilities

# Directness patterns (from Zero spec)
HEDGING_PATTERNS = [
    r'\bmight\b', r'\bperhaps\b', r'\bpossibly\b', r'\bprobably\b',
    r'\bseems? (like|to)\b', r'\bappears? to\b',
    r'\bI think\b', r'\bI believe\b', r'\bmaybe\b',
    r'\bcould potentially\b',
]

ABSTENTION_PHRASES = [
    r"can't determine this without",
    r"cannot be solved because",
    r"insufficient data",
    r"not enough information",
    r"would need.*to (continue|solve|verify)",
]

# Semantic model for crypto / forensics partial-match scoring
_sem_model = None
def get_sem_model():
    global _sem_model
    if _sem_model is None:
        _sem_model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
    return _sem_model

def score_correctness(completion: str, problem: dict) -> float:
    """
    Returns 1.0 (correct), 0.5 (partially correct / semantic match), or 0.0 (wrong).
    """
    answer = problem['answer'].lower().strip()
    completion_lower = completion.lower()

    # Fast path: exact substring match
    if answer in completion_lower:
        return 1.0

    if problem['answer_type'] == 'semantic':
        model = get_sem_model()
        emb_c = model.encode(completion, convert_to_tensor=True)
        emb_a = model.encode(problem['answer'], convert_to_tensor=True)
        sim = util.cos_sim(emb_c, emb_a).item()
        if sim >= 0.82:
            return 1.0
        elif sim >= 0.65:
            return 0.5

    return 0.0

def score_directness(completion: str) -> dict:
    """
    Returns directness metrics:
    - hedge_count: number of hedging patterns found
    - is_abstention: model explicitly said it can't solve this
    - directness_score: 1.0 (clean) down to 0.0 (heavily hedged)
    """
    hedge_count = sum(1 for p in HEDGING_PATTERNS if re.search(p, completion, re.IGNORECASE))
    is_abstention = any(re.search(p, completion, re.IGNORECASE) for p in ABSTENTION_PHRASES)
    directness_score = max(0.0, 1.0 - (hedge_count * 0.15))
    return {
        'hedge_count': hedge_count,
        'is_abstention': is_abstention,
        'directness_score': round(directness_score, 3),
    }

print("Scoring utilities ready.")

In [ ]:
# Cell 5 — Inference function

SYSTEM_PROMPT = """You are a security researcher analyzing CTF challenges. 
Reason through the problem and give a direct, precise answer. 
No hedging. No uncertainty padding. If you can solve it, solve it. 
If you genuinely cannot determine the answer from the given information, say exactly what is missing."""

def load_model(model_name: str):
    print(f"\nLoading {model_name}...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    print(f"  Loaded. VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    return tokenizer, model

def run_inference(tokenizer, model, prompt: str, max_new_tokens: int = 512) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,       # greedy — deterministic, reproducible
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the new tokens
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def unload_model(model):
    del model
    torch.cuda.empty_cache()
    print("  Model unloaded.")

print("Inference functions ready.")

In [ ]:
# Cell 6 — Main eval loop
# Runs all three models sequentially. Loads, evaluates, unloads before next.
# Saves results to CSV after each model in case of session interruption.

MODELS = [
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-3B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
]

all_results = []

for model_name in MODELS:
    model_label = model_name.split("/")[-1]  # e.g. Qwen2.5-3B-Instruct
    tokenizer, model = load_model(model_name)

    for problem in BENCHMARK:
        print(f"  [{model_label}] {problem['id']}... ", end="", flush=True)

        completion = run_inference(tokenizer, model, problem['prompt'])
        correctness = score_correctness(completion, problem)
        directness = score_directness(completion)

        result = {
            'model': model_label,
            'problem_id': problem['id'],
            'category': problem['category'],
            'difficulty': problem['difficulty'],
            'correctness': correctness,
            'directness_score': directness['directness_score'],
            'hedge_count': directness['hedge_count'],
            'is_abstention': directness['is_abstention'],
            'completion': completion,
            'expected_answer': problem['answer'],
        }
        all_results.append(result)

        status = "✓" if correctness == 1.0 else ("~" if correctness == 0.5 else "✗")
        print(f"{status} (correct={correctness}, direct={directness['directness_score']})")

    # Save checkpoint after each model
    df_checkpoint = pd.DataFrame(all_results)
    df_checkpoint.to_csv(f"/kaggle/working/zero_phase1_{model_label}.csv", index=False)
    print(f"  Checkpoint saved for {model_label}.")

    unload_model(model)

# Final combined CSV
df = pd.DataFrame(all_results)
df.to_csv("/kaggle/working/zero_phase1_results.csv", index=False)
print(f"\nDone. {len(df)} results saved to zero_phase1_results.csv")

In [ ]:
# Cell 7 — Results analysis
# Summary tables. This is what goes into the Phase 1 findings.

df = pd.read_csv("/kaggle/working/zero_phase1_results.csv")

print("=" * 60)
print("ZERO — PHASE 1 BASELINE RESULTS")
print("=" * 60)

# Overall score per model
print("\n── Overall correctness per model ──")
overall = df.groupby('model').agg(
    solve_rate=('correctness', 'mean'),
    avg_directness=('directness_score', 'mean'),
    avg_hedges=('hedge_count', 'mean'),
    abstentions=('is_abstention', 'sum'),
).round(3)
print(overall.to_string())

# Per category breakdown
print("\n── Solve rate by model × category ──")
cat_breakdown = df.groupby(['model', 'category'])['correctness'].mean().unstack().round(3)
print(cat_breakdown.to_string())

# Per difficulty breakdown
print("\n── Solve rate by model × difficulty ──")
diff_breakdown = df.groupby(['model', 'difficulty'])['correctness'].mean().unstack().round(3)
print(diff_breakdown.to_string())

# Individual problem breakdown
print("\n── Per-problem results ──")
pivot = df.pivot_table(
    index='problem_id',
    columns='model',
    values='correctness'
).round(3)
print(pivot.to_string())

print("\n── Sample completions (first problem, all models) ──")
first_id = BENCHMARK[0]['id']
for _, row in df[df['problem_id'] == first_id].iterrows():
    print(f"\n[{row['model']}]")
    print(row['completion'][:500])
    print("-" * 40)

## After running

The output CSV (`zero_phase1_results.csv`) contains every completion and score. Download it and commit the summary tables to the repo under `results/phase1_baseline.md`.

**What to look for:**
- Does correctness scale with model size, or is 1.5B surprisingly capable?
- Which categories are hardest at baseline? (Expected: crypto and RE harder than web)
- How hedgy are the base models? High hedge count = more work for directness training
- Any spontaneous abstentions? (Interesting signal either way)

These findings directly inform Phase 2 dataset construction priorities.